## 1. Importing libraries & packages

In [1]:
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt

## 2. Load cleaned dataset & quick sanity check

In [2]:
df = pd.read_csv("data/processed/train_clean.csv")
df.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
2,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
3,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0
4,5,D012,HARD,Saudi Arabian Grand Prix,2022,0,35,2,26.0,5,93.734,-9.878,-98.041,0.500000,-3.0,1.0


In [3]:
print((f"No. of rows: {df.shape[0]}") + (f" \nNo. of columns: {df.shape[1]}"))

No. of rows: 432376 
No. of columns: 16


In [4]:
df.describe()

,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
count,432376.000000,432376.000000,432376.000000,432376.000000,432376.000000,432376.000000,432376.000000,432376.000000,432376.000000,432376.000000,432376.000000,432376.000000,432376.000000
mean,219526.782493,2023.503495,0.133402,22.942208,1.786942,14.081477,9.648778,90.893186,-3.064427,-23.102136,0.335976,0.101708,0.193676
std,126757.371627,1.016495,0.340010,16.959407,0.956548,9.781057,5.277386,11.415182,12.419450,47.616349,0.253944,4.000709,0.395179
min,0.000000,2022.000000,0.000000,1.000000,1.000000,1.000000,1.000000,67.694000,-59.959000,-200.000000,0.012821,-18.000000,0.000000
25%,109743.750000,2023.000000,0.000000,9.000000,1.000000,6.000000,5.000000,82.715000,-8.857000,-44.964250,0.128205,-1.000000,0.000000
50%,219520.500000,2023.000000,0.000000,19.000000,2.000000,12.000000,10.000000,90.631000,-0.265000,-20.891000,0.263889,0.000000,0.000000
75%,329298.250000,2024.000000,0.000000,36.000000,2.000000,20.000000,14.000000,98.541000,0.122000,-6.061000,0.512821,2.000000,0.000000
max,439139.000000,2025.000000,1.000000,78.000000,8.000000,77.000000,20.000000,199.512000,59.829000,175.508000,1.000000,18.000000,1.000000


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 432376 entries, 0 to 432375
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      432376 non-null  int64  
 1   Driver                  432376 non-null  str    
 2   Compound                432376 non-null  str    
 3   Race                    432376 non-null  str    
 4   Year                    432376 non-null  int64  
 5   PitStop                 432376 non-null  int64  
 6   LapNumber               432376 non-null  int64  
 7   Stint                   432376 non-null  int64  
 8   TyreLife                432376 non-null  float64
 9   Position                432376 non-null  int64  
 10  LapTime (s)             432376 non-null  float64
 11  LapTime_Delta           432376 non-null  float64
 12  Cumulative_Degradation  432376 non-null  float64
 13  RaceProgress            432376 non-null  float64
 14  Position_Change         432376 

## 3. Feature Engineering Roadmap

- This notebook builds on `data/processed/train_clean.csv`. Each engineered feature (or removal 
decision) below is traced back to a specific EDA finding — no feature is added without 
evidence-based justification.

| # | Feature / Action | EDA Evidence | Hypothesis | Status |
|---|---|---|---|---|
| 1 | Drop `LapNumber` | `LapNumber` and `RaceProgress` show 0.9645 correlation (verified) — near-identical signal. `02_EDA` §3.2.1 and §3.2.7 both showed similar separation patterns vs. `PitNextLap`. | `RaceProgress` is the better-generalizing version (normalized for race length across circuits), so it's kept; `LapNumber` is redundant and risks multicollinearity. | ✅ Decided
| 2 | `TyreLife_Relative` | HARD compound had highest pit rate (~35%), WET lowest — compounds clearly differ in degradation/usage behavior (`02_EDA` §3.1.1). Raw `TyreLife` treats all compounds identically, but the same tire age means different things per compound. | The *relative* wear of a tire — `TyreLife / avg_TyreLife_for_that_compound` — is a stronger pit-likelihood signal than raw `TyreLife` alone, since it accounts for compound-specific degradation speed. | ✅ Built — validation deferred to modelling |

### 3.1 `LapNumber` and `RaceProgress` — Correlation & Potential Multicolinearity 

***Hypothesis derived from `02_EDA_bivariate_analysis` §3.2.1 & §3.2.7***

**The reasoning:-**

- `LapNumber` and `RaceProgress` showed near-identical patterns when plotted against `PitNextLap` in `02_EDA_bivariate_analysis` (§3.2.1 and §3.2.7) — both captured the same underlying idea: as a race progresses, the likelihood of a pit stop changes in a similar way.

- This visual similarity raised a hypothesis worth testing empirically rather than assuming: are these two 
features actually redundant, or do they just look similar by coincidence?

In [6]:
correlation = df['LapNumber'].corr(df['RaceProgress'])
print(f"Correlation between LapNumber and RaceProgress: {correlation:.4f}")

Correlation between LapNumber and RaceProgress: 0.9645


- A correlation check confirmed the hypothesis — `LapNumber` and `RaceProgress` showed a correlation of **0.9645**, indicating they carry almost the same information.

**Why redundancy matters here — multicollinearity:**

- Keeping two highly correlated features in a model doesn't add new signal; it introduces **multicollinearity**, which affects different model types in different ways:

- **Linear/logistic models** assign weights to each feature based on its contribution to the prediction. When two features carry near-identical information, the model has no clean way to decide which one "deserves" the weight — it ends up splitting credit between them, making the learned coefficients unstable and harder to interpret.

- **Tree-based models** (Random Forest, XGBoost) are more robust to this, but feature importance scores still get diluted — since the model can split on either feature to achieve a similar gain, the true importance of the underlying signal gets artificially divided across two columns instead of concentrated in one.

**The Decision:-**

- **`LapNumber`** is dropped. **`RaceProgress`** is retained as the sole representation of race-stage progression going forward.

- `RaceProgress` is the strictly better-generalizing version of this signal. `LapNumber` is a raw integer whose meaning depends on the total length of the race — "lap 25" represents very different race stages in a 50-lap race versus a 70-lap race.

- `RaceProgress`, computed as `LapNumber / TotalLaps`, normalizes for this, expressing race position as a consistent 0–1 ratio regardless of circuit. This makes it the more robust, transferable feature across different races.

In [7]:
df_clean = df.drop(columns=['LapNumber'])
print(f"Dropped LapNumber. Remaining columns: {df_clean.shape[1]}")
print(df_clean.columns.tolist())

Dropped LapNumber. Remaining columns: 15
['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'PitNextLap']


### 3.2 `TyreLife_Relative` — Tire Age Relative to Compound-Specific Lifespan

***Hypothesis derived from `02_EDA_bivariate_analysis` §3.1.1***

**The reasoning:**

- EDA showed that pit rate varies meaningfully by tire compound — **HARD** compounds had the highest pit-next-lap rate (~35%), while **WET** had the lowest. This tells us compounds don't degrade (or get used) at the same rate.

- A direct consequence: `TyreLife = 18` laps means something very different for a SOFT tire (likely well past its typical usable life) versus a HARD tire (likely still mid-life).

- The raw `TyreLife` feature treats all compounds identically — it has no way to express "18 laps is old for this compound" vs. "18 laps is still fresh for that compound."

- Tree-based models *can* learn this interaction implicitly by splitting on `Compound` and then `TyreLife` 
separately, but explicitly engineering the relationship makes the signal available immediately, rather than relying on the model to discover it.

**The feature:**

TyreLife_Relative = TyreLife / average_TyreLife_for_that_Compound



A value > 1 means the tire has exceeded its compound's typical lifespan (stronger pit signal); 
a value < 1 means it's still within typical range.

**A data leakage note:**

- The compound averages used to build this feature should, in principle, be calculated only from the *training* split. Once train/test splitting happens in the modelling notebook — calculating them on the full dataset (including future validation/test rows) risks leaking information the model shouldn't have access to during training.

- For this exploratory notebook, the averages are computed on the full `df` to validate the concept, but the feature is built via a reusable function (`add_tyrelife_relative`) so it can be recalculated correctly on the training fold only, later.

In [8]:
def add_tyrelife_relative(df, reference_df=None):
    """
    Adds TyreLife_Relative = TyreLife / avg_TyreLife_for_that_compound.
    
    reference_df: the dataframe to CALCULATE the compound averages from. 
                   In modelling, this should be the TRAINING split only,
                   to avoid data leakage into validation/test sets.
                   If None, uses df itself (fine for this exploratory notebook,
                   NOT fine once train/test splitting begins).
    """
    if reference_df is None:
        reference_df = df
    
    compound_avg = reference_df.groupby('Compound')['TyreLife'].mean()
    
    df = df.copy()
    df['TyreLife_Relative'] = df['TyreLife'] / df['Compound'].map(compound_avg)
    
    return df, compound_avg

In [9]:
df_clean, compound_avg_lookup = add_tyrelife_relative(df)

print(compound_avg_lookup)
print()
df_clean[['Compound', 'TyreLife', 'TyreLife_Relative']].head(10)

Compound
HARD            17.760908
INTERMEDIATE    13.296595
MEDIUM          11.803840
SOFT            11.155726
WET              9.691847
Name: TyreLife, dtype: float64



,Compound,TyreLife,TyreLife_Relative
0,HARD,39.0,2.195834
1,HARD,22.0,1.238675
2,MEDIUM,2.0,0.169436
3,HARD,6.0,0.337821
4,HARD,26.0,1.463889
5,MEDIUM,16.0,1.355491
6,HARD,9.0,0.506731
7,MEDIUM,4.0,0.338873
8,MEDIUM,2.0,0.169436
9,HARD,10.0,0.563034


**A validity check that changed the plan:**

- Inspecting the per-compound averages revealed `WET` had both the smallest sample size (1,337 rows, 0.3% of data) and, counter-intuitively, the *lowest* average `TyreLife` — despite EDA showing WET had the *lowest* pit rate (drivers stay out longest). 

- Investigating this contradiction: pit decisions on WET tires are driven primarily by **weather and strategy** (e.g. waiting for the track to dry before switching to slicks), not by pure tire degradation the way dry compounds are. This means "relative tire age" is a less meaningful — and statistically less reliable, given the tiny sample — signal specifically for WET.

**Decision:** 
- `TyreLife_Relative` is retained for all compounds (no special-casing in the calculation), but raw `Compound` is also kept as an input feature so models can learn to treat **WET** rows differently on their own, rather than relying solely on a fragile relative-tire-age number for that compound.
- A dedicated wet-weather model is a reasonable future extension but is out of scope for this iteration.

**Validation deferred:**
- Whether `TyreLife_Relative` outperforms raw `TyreLife` + `Compound` will be tested empirically via Mutual Information and Random Forest feature importance during modelling — not assumed here.

In [10]:
df_clean.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,TyreLife_Relative
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0,2.195834
1,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0,1.238675
2,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0,0.169436
3,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0,0.337821
4,5,D012,HARD,Saudi Arabian Grand Prix,2022,0,35,2,26.0,5,93.734,-9.878,-98.041,0.500000,-3.0,1.0,1.463889
